# Implied Volatility Surface Builder
### SPY + QQQ + IWM Options | Gaussian Process + MLP | Arbitrage Checks

This notebook walks through the full pipeline step by step:
1. Fetch live options data
2. Extract implied volatility via Black-Scholes inversion
3. Fit GP and MLP surfaces
4. Check arbitrage conditions
5. Visualize results

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split

sys.path.insert(0, os.path.abspath('.'))
np.random.seed(42)
print('Libraries loaded ✓')

Libraries loaded ✓


## Step 1 — Fetch Options Data

We pull live delayed options chains from SPY, QQQ, and IWM via yfinance.
All strikes are converted to **moneyness space** (K/S) so data from different
underlyings can be combined meaningfully.

In [ ]:
from data.fetch_options import fetch_spy_options

S, r, df_raw = fetch_spy_options(["SPY", "QQQ", "IWM"])
print(f"\nSpot (SPY): ${S:.2f}")
print(f"Risk-free rate: {r*100:.2f}%")
print(f"Total contracts: {len(df_raw)}")
df_raw.head(10)

[Data] Risk-free rate: 3.59%
[Data] SPY spot: $711.17
[Data] SPY raw contracts: 8162
[Data] QQQ spot: $660.37
[Data] QQQ raw contracts: 6953
[Data] IWM spot: $272.52
[Data] IWM raw contracts: 3418

[Data] Total contracts: 12780
underlying
IWM    2374
QQQ    4828
SPY    5578
[Data] Expirations: 30

Spot (SPY): $711.17
Risk-free rate: 3.59%
Total contracts: 12780


,strike,expiry,tte,mid_price,option_type,moneyness,underlying,spot,yf_iv
0,525.0,2026-04-30,0.0,186.565,call,0.738215,SPY,711.174988,1.846680
1,530.0,2026-04-30,0.0,181.585,call,0.745246,SPY,711.174988,1.808595
2,535.0,2026-04-30,0.0,176.565,call,0.752276,SPY,711.174988,1.745118
3,540.0,2026-04-30,0.0,171.520,call,0.759307,SPY,711.174988,1.666017
4,545.0,2026-04-30,0.0,166.965,call,0.766337,SPY,711.174988,1.830079
5,550.0,2026-04-30,0.0,162.040,call,0.773368,SPY,711.174988,1.802247
6,555.0,2026-04-30,0.0,156.970,call,0.780399,SPY,711.174988,1.723634
7,560.0,2026-04-30,0.0,152.065,call,0.787429,SPY,711.174988,1.702150
8,565.0,2026-04-30,0.0,146.875,call,0.794460,SPY,711.174988,1.583010
9,570.0,2026-04-30,0.0,142.115,call,0.801491,SPY,711.174988,1.609377


In [ ]:
# Distribution of contracts by underlying and expiry
import plotly.express as px

fig = px.histogram(df_raw, x='tte', color='underlying', nbins=30,
                   title='Contract Distribution by Time to Expiry',
                   labels={'tte': 'Time to Expiry (years)'},
                   template='plotly_dark')
fig.show()

## Step 2 — Black-Scholes IV Extraction

For each contract we solve numerically for σ such that:

$$BS(\sigma) - C_{market} = 0$$

Using **Brent's method** — a bracketing root-finder guaranteed to converge.

The Black-Scholes call formula:
$$C = S \cdot N(d_1) - K e^{-rT} \cdot N(d_2)$$
$$d_1 = \frac{\ln(S/K) + (r + \sigma^2/2)T}{\sigma\sqrt{T}}, \quad d_2 = d_1 - \sigma\sqrt{T}$$

In [ ]:
# Quick sanity check first
from iv.extractor import bs_call, extract_iv

true_sigma = 0.20
price = bs_call(S=100, K=100, T=0.25, r=0.05, sigma=true_sigma)
recovered = extract_iv(price, S=100, K=100, T=0.25, r=0.05, option_type='call')
print(f"True σ:      {true_sigma:.6f}")
print(f"Recovered σ: {recovered:.6f}")
print(f"Error:       {abs(recovered - true_sigma):.2e}")

True σ:      0.200000
Recovered σ: 0.200000
Error:       8.01e-09


In [ ]:
from iv.extractor import extract_iv_surface

df = extract_iv_surface(df_raw, S, r)
print(f"\nClean IV dataset: {len(df)} contracts")
print(f"IV range: {df['iv'].min():.1%} – {df['iv'].max():.1%}")
print(f"Median IV: {df['iv'].median():.1%}")

[IV] Extracting IV for 12780 contracts...


Brent root-finding: 100%|██████████| 12780/12780 [00:43<00:00, 290.91it/s]

[IV] Success: 7431/7431 | Range: 5.1%–54.1%

Clean IV dataset: 7431 contracts
IV range: 5.1% – 54.1%
Median IV: 23.6%


In [ ]:
# IV distribution by underlying
fig = px.box(df, x='underlying', y='iv', color='underlying',
             title='IV Distribution by Underlying',
             labels={'iv': 'Implied Volatility'},
             template='plotly_dark')
fig.update_yaxes(tickformat='.0%')
fig.show()

In [ ]:
# IV smile — raw market data scatter
fig = px.scatter(df, x='moneyness', y='iv', color='underlying',
                 facet_col='option_type', opacity=0.6,
                 title='Raw IV Smile — Market Data',
                 labels={'iv': 'Implied Volatility', 'moneyness': 'Moneyness (K/S)'},
                 template='plotly_dark')
fig.update_yaxes(tickformat='.0%')
fig.add_vline(x=1.0, line_dash='dot', line_color='orange', annotation_text='ATM')
fig.show()

## Step 3a — Gaussian Process Surface

GP models IV as a random function with prior:
$$\sigma(K,T) \sim \mathcal{GP}(0, k(x, x'))$$

Kernel: **Anisotropic RBF** — separate length scales for moneyness and TTE,
plus a WhiteKernel for observation noise.

Key advantage: **probabilistic output** — gives mean + uncertainty band.

In [ ]:
from models.gp_surface import IVSurfaceGP

df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)

gp = IVSurfaceGP(n_restarts=5)
gp.fit(df_train)

gp_metrics = gp.evaluate(df_test)
print(f"\nGP Test Results:")
print(f"  MAE:  {gp_metrics['MAE']:.4f} ({gp_metrics['MAE']*100:.2f}% IV)")
print(f"  RMSE: {gp_metrics['RMSE']:.4f}")
print(f"  R²:   {gp_metrics['R2']:.4f}")

[GP] Subsampled to 500 points
[GP] Fitting on 500 points...
[GP] Train R²: 0.8552

GP Test Results:
  MAE:  0.0266 (2.66% IV)
  RMSE: 0.0497
  R²:   0.7719


## Step 3b — MLP Surface

Architecture: `Input(3) → 64 → 128 → 64 → Softplus(1)`

- **Softplus** output guarantees IV > 0 smoothly
- **BatchNorm** + **Dropout** for regularization  
- **Cosine annealing** LR schedule
- **Early stopping** on validation loss

In [ ]:
from models.mlp_surface import IVSurfaceMLP

mlp = IVSurfaceMLP(epochs=300, patience=30)
mlp.fit(df_train)

mlp_metrics = mlp.evaluate(df_test)
print(f"\nMLP Test Results:")
print(f"  MAE:  {mlp_metrics['MAE']:.4f} ({mlp_metrics['MAE']*100:.2f}% IV)")
print(f"  RMSE: {mlp_metrics['RMSE']:.4f}")
print(f"  R²:   {mlp_metrics['R2']:.4f}")

[MLP] Training on cpu | 5052 train, 892 val
[MLP] Epoch   1 | train: 0.798263 | val: 0.609484
[MLP] Epoch  50 | train: 0.495281 | val: 0.502634
[MLP] Early stop at epoch 91 | best val: 0.499074

MLP Test Results:
  MAE:  0.0534 (5.34% IV)
  RMSE: 0.0720
  R²:   0.5218


In [ ]:
import importlib
import models.mlp_surface as mlp_mod
importlib.reload(mlp_mod)
from models.mlp_surface import IVSurfaceMLP

mlp = IVSurfaceMLP(epochs=300, patience=30)
mlp.fit(df_train)

In [ ]:
# Training loss curve
fig = go.Figure()
fig.add_trace(go.Scatter(y=mlp.history['train_loss'], name='Train Loss',
                         line=dict(color='#636EFA')))
fig.add_trace(go.Scatter(y=mlp.history['val_loss'], name='Val Loss',
                         line=dict(color='#EF553B')))
fig.update_layout(title='MLP Training History', xaxis_title='Epoch',
                  yaxis_title='MSE Loss', template='plotly_dark')
fig.show()

AttributeError: 'IVSurfaceMLP' object has no attribute 'history'

## Step 4 — Build and Visualize IV Surfaces

In [ ]:
m_grid = np.linspace(0.80, 1.20, 60)
t_grid = np.linspace(7/365, 1.0, 40)

iv_gp, iv_gp_std = gp.predict_grid(m_grid, t_grid)
iv_mlp           = mlp.predict_grid(m_grid, t_grid)

print(f"GP surface shape: {iv_gp.shape}")
print(f"IV range (GP):  {iv_gp.min():.1%} – {iv_gp.max():.1%}")
print(f"IV range (MLP): {iv_mlp.min():.1%} – {iv_mlp.max():.1%}")

In [ ]:
# 3D Surface — GP
from plotly.subplots import make_subplots

M, T = np.meshgrid(m_grid, t_grid, indexing='ij')
fig = make_subplots(rows=1, cols=2, specs=[[{'type':'surface'},{'type':'surface'}]],
                    subplot_titles=['Gaussian Process', 'MLP'])
fig.add_trace(go.Surface(x=M, y=T*365, z=iv_gp*100, colorscale='Viridis',
                         colorbar=dict(x=0.46, title='IV %')), row=1, col=1)
fig.add_trace(go.Surface(x=M, y=T*365, z=iv_mlp*100, colorscale='Plasma',
                         colorbar=dict(x=1.02, title='IV %')), row=1, col=2)
fig.update_layout(title='SPY IV Surface — GP vs MLP', height=600, template='plotly_dark',
                  scene=dict(xaxis_title='Moneyness', yaxis_title='Days', zaxis_title='IV %'),
                  scene2=dict(xaxis_title='Moneyness', yaxis_title='Days', zaxis_title='IV %'))
fig.show()

In [ ]:
# GP Uncertainty Band at fixed maturity
idx = 10
T_val = t_grid[idx]
iv_slice  = iv_gp[:, idx] * 100
std_slice = iv_gp_std[:, idx] * 100

fig = go.Figure()
fig.add_trace(go.Scatter(x=m_grid, y=iv_slice+std_slice, line=dict(width=0), showlegend=False))
fig.add_trace(go.Scatter(x=m_grid, y=iv_slice-std_slice, line=dict(width=0),
                         fill='tonexty', fillcolor='rgba(99,110,250,0.2)', name='±1σ'))
fig.add_trace(go.Scatter(x=m_grid, y=iv_slice, line=dict(color='#636EFA', width=2.5), name='GP mean'))
fig.add_vline(x=1.0, line_dash='dot', line_color='orange', annotation_text='ATM')
fig.update_layout(title=f'GP Uncertainty Band — T={T_val*365:.0f} days',
                  xaxis_title='Moneyness (K/S)', yaxis_title='IV (%)',
                  template='plotly_dark', height=450)
fig.show()

## Step 5 — Arbitrage Checks

**Calendar spread:** Total variance $w(K,T) = \sigma^2 \cdot T$ must be non-decreasing in $T$.

**Butterfly spread:** $w(x,T)$ must be convex in log-moneyness $x = \log(K/S)$.

In [ ]:
from arbitrage.checks import run_arbitrage_checks

print("GP Surface:")
gp_arb = run_arbitrage_checks(iv_gp, m_grid, t_grid)

print("\nMLP Surface:")
mlp_arb = run_arbitrage_checks(iv_mlp, m_grid, t_grid)

## Final Results

In [ ]:
results = pd.DataFrame({
    'Model': ['Gaussian Process', 'MLP'],
    'MAE':   [f"{gp_metrics['MAE']:.4f}", f"{mlp_metrics['MAE']:.4f}"],
    'RMSE':  [f"{gp_metrics['RMSE']:.4f}", f"{mlp_metrics['RMSE']:.4f}"],
    'R²':    [f"{gp_metrics['R2']:.4f}", f"{mlp_metrics['R2']:.4f}"],
    'Calendar Violations':  [gp_arb.calendar_violations,  mlp_arb.calendar_violations],
    'Butterfly Violations': [gp_arb.butterfly_violations, mlp_arb.butterfly_violations],
})
results